In [ ]:
pip install segmentation-models-pytorch

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os
import numpy as np 
import pandas as pd 
import cv2 as cv
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision.transforms import v2, InterpolationMode
from torchvision import transforms
import torchvision.transforms as transforms
from torchvision.ops import sigmoid_focal_loss
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
import segmentation_models_pytorch as smp
import torch.optim as optim
from torchvision.io import read_image, ImageReadMode
import torch.nn as nn
import random
import torch
import time
from tqdm import tqdm
from torchmetrics.classification import (BinaryPrecision,BinaryRecall,BinaryF1Score,BinaryJaccardIndex)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
"""for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))"""
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

### The dataset contains 290 RGB images with corresponding binary segmentation masks. Original image resolution varies across samples (ex, 551×893), therefore resizing is required before training.

In [ ]:
# Quick view of how the data is stored 
root_path = "/kaggle/input/flood-area-segmentation/"

img_mask_mapping_df = pd.read_csv(root_path + "metadata.csv")
display(img_mask_mapping_df.head())
img_mask_mapping_df.info()
img = cv.imread(root_path+"/Image/0.jpg")
mask = cv.imread(root_path+"/Mask/0.png", cv.IMREAD_GRAYSCALE)
print("Image shape: ",img.shape)
print("Mask shape: ",mask.shape)

### Dataset Class Design
* Implemented a custom READ_FLOOD_AREA_DATA_SET class to handle:
* Image loading
* Target preprocessing
* Supervision mask generation
* Optional data augmentation
* Different operational modes (train / evaluation / test)
### Image Processing
For each sample:
* Images are loaded in RGB format using read_image to ensure compatibility with PyTorch.
* Pixel values are normalized to [0, 1].
* All images are resized to a fixed resolution ex:(224×224) to:
* *  Reduce computational cost
* *  Ensure consistent input size
* ImageNet normalization is applied to match the pretrained encoder distribution.
### Target ( the ground truth segmentation mask) Processing
* Target masks are loaded as grayscale images.
* Pixel values are binarized:
* *  0 → Background, 1 → Water
* Masks are resized to match image resolution
* Final shape: (H, W).
### Mask Generation
* To simulate partial supervision, generate a binary mask indicating which pixels contribute to the loss:
* * 1 → Pixel used in loss, 0 → Pixel ignored
* Two strategies are supported:
* * Random Sampling:
* * * Randomly drops a percentage of pixels from the entire mask.
* * Class-Balanced Sampling
* * *  Drops pixels separately from: Water class and Background class
* * *  This allows better control of class imbalance during the experiment.
    *  
### Data Augmentation (Optional)
* In (training mode only):
* Photometric augmentation (applied to image only): Random brightness & contrast
* Geometric augmentation (applied to both image and target), to ensures spatial alignment between image and the target:
* *  Random horizontal flip

### Modes:
* train --> image, target, mask
* evaluation --> image, full target
* test --> image only

In [ ]:
class READ_FLOOD_AREA_DATA_SET(Dataset):
   def __init__(self, df, root_url, drop_pixels_percentage=None, img_size=(224, 224), augmentation_mode=False,mode="train", 
               mask_strategy='random'):
    self.df = df
    self.root_url = root_url
    self.drop_pixels_percentage = drop_pixels_percentage
    self.mode = mode
    self.augmentation_mode = augmentation_mode
    self.mask_strategy = mask_strategy
    # Resize is always applied to enforce a fixed input resolution and reduce computational cost
    self.img_resize = v2.Resize(img_size, antialias=True, interpolation=InterpolationMode.BICUBIC)
    self.target_resize = v2.Resize(img_size, antialias=False, interpolation=InterpolationMode.NEAREST_EXACT) 
    if self.augmentation_mode:
        # Photometric augmentations (image only)
        self.img_only_aug = v2.ColorJitter(brightness=(0.7, 1.3),contrast=(0.8, 1.2))

        # Geometric augmentations (shared between image and target)
        self.img_target_aug = v2.Compose([
                                          v2.RandomHorizontalFlip(p=0.5)
                                         #,v2.RandomRotation(degrees=30,interpolation=v2.InterpolationMode.NEAREST)
                                         ])

    # Normalization uses ImageNet statistics to match the pretrained encoder
    self.img_normalization = v2.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
       
   def __len__(self): 
        return len(self.df)

   def class_balanced_sampling_mask(self, target):
       """
       Drop pixels in a class-balanced manner, where a fixed percentage of water and background pixels
       are randomly ignored
       Possible enhancment: drop_water_percentage, drop_background_percentage
       """
       mask = torch.ones_like(target)
       # indices per class
       water_idx = torch.where(target == 1)
       background_idx = torch.where(target == 0)

       num_water_pixels = water_idx[0].shape[0]
       num_background_pixels = background_idx[0].shape[0]

       if num_water_pixels == 0 or num_background_pixels == 0:
           return mask

       # number of pixels to drop per class
       num_drop_water_pixels = int(num_water_pixels * self.drop_pixels_percentage)
       num_drop_background_pixels = int(num_background_pixels * self.drop_pixels_percentage)

       num_drop_water_pixels = max(1, num_drop_water_pixels)
       num_drop_background_pixels = max(1, num_drop_background_pixels)

       # randomly drop water pixels
       shuffled_water_idx = torch.randperm(num_water_pixels)
       water_drop_idx = shuffled_water_idx[:num_drop_water_pixels]

       mask[water_idx[0][water_drop_idx], water_idx[1][water_drop_idx]] = 0
       
       # randomly drop background pixels
       shuffled_background_idx = torch.randperm(num_background_pixels)
       background_drop_idx = shuffled_background_idx[:num_drop_background_pixels]
       
       mask[background_idx[0][background_drop_idx],background_idx[1][background_drop_idx]] = 0
       
       return mask    

    
   def get_random_mask(self, target_shape):
       """
       Create a mask by randomly sampling a percentage
       of all labeled pixels for both background and water
       (Randomly keep (1 - mask_percentage) of all pixels and ignore the rest)
       # mask values: 1 = used in loss, 0 = ignored
       """
       H, W = target_shape
       mask = torch.ones((H,W))
       num_pixels = H*W
       num_drop_pixels = int(num_pixels * self.drop_pixels_percentage)
       num_drop_pixels = max(1, num_drop_pixels)

       # randomly select pixels indices to drop
       drop_indices = torch.randperm(num_pixels)[:num_drop_pixels]

       # flatten mask and drop
       mask.view(-1)[drop_indices] = 0
       
       return mask
        
   def __getitem__(self, idx):
       """
       Returns:
       self.mode --> train, validation, test
       train -or validation- mode: image, target, mask
       evaluation mode: image, target
       test mode : image only
       """
       # ---------- Load image ----------
       img_path = os.path.join(self.root_url, "Image", self.df.iloc[idx,0])
       img = read_image(img_path, mode=ImageReadMode.RGB).float() / 255.0  # (3,H,W), RGB
       img = self.img_resize(img)

       if self.mode == "test":
           # test / inference mode
           return {"img": self.img_normalization(img)}
           
       #---------- Load target ----------
       target_path = os.path.join(self.root_url, "Mask", self.df.iloc[idx, 1]) 
       target = read_image(target_path) # (1,H,W)
       target = self.target_resize(target)
       target = target.squeeze(0)  # (H, W)
       target = (target > 0).float() # binarize (water / background), float to be compatible with the loss fun 
       if self.mode == "evaluation":
           return {"img": self.img_normalization(img), "target": target}

       # ______________ Augmentation(op) ______________  
       if self.mode == "train" and self.augmentation_mode:
           # spatial transforms must be shared, on the fly method
           img = self.img_only_aug(img)
           img, target = self.img_target_aug(img, target)
           
       if self.mask_strategy == "random":
           mask = self.get_random_mask(target.shape)
       elif self.mask_strategy == "balanced":
           mask = self.class_balanced_sampling_mask(target)    

                  
       return {"img": self.img_normalization(img), "target": target, "mask": mask}

           

print("Done")

In [ ]:
def img_visualization(input_imgs_dict, title=None):
    """
    input_imgs_dict --> {"Image":img, "Mask":mask, "Target": target, "Output": output}
    All inputs are torch Tensors.
    """
    def denormalize(img_tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
        return (img_tensor * std + mean).clamp(0, 1)
        
    ncols = len(input_imgs_dict)
    fig, axes = plt.subplots(nrows=1, ncols=ncols, figsize=(15, 5))
    if title:
        fig.suptitle(title, fontsize=14);
        
    for i, (key, value) in enumerate(input_imgs_dict.items()):
        #print(f"I'm {key}, shape = {tuple(value.shape)}")
        img = value.detach().cpu()
        if img.ndim == 3 and img.shape[0] == 3:
            img = denormalize(img)
            img = img.permute(1, 2, 0) # RGB image: (3, H, W) ---> (H, W, 3)
            axes[i].imshow(img)
            
        elif img.ndim == 3 and img.shape[0] == 1:
            img = img.squeeze(0)  # (1, H, W) ---> (H, W)
            axes[i].imshow(img, cmap="gray")

        elif img.ndim == 2:
            axes[i].imshow(img, cmap="gray")
        else:
            raise ValueError(f"unsupported shape for visualization: {img.shape}")
            
        axes[i].set_title(key)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()   

print("Done^^")

# Spatial Focal Loss:
* To solve the main challange of segmentation problem whic is class imbalance as background pixels dominate most images, and lack of accured labeled pixels so, will use focal loss + mask
During training, only a subset of pixels contribute to the loss (using masks).
* Why not a standard Binary Cross Entropy?
Becasue it will treat easy and hard examples equally, and the loss would be dominated by background pixels, so focal loss would give less weights to easy examples classification (background) to influance the gradients values, and using the mask would adabot the network to deal with lack in segmented/labeled pixels 

# Focal Loss Overview:
* The idea is to dowen weight the influance of easy examples/ common class ex background pixels, that give the rare/hard examples ex flood pixels the chance to have more influance to the loss value 
Same BCE equation with extra 2 hyper paramters
* α ∈ [0,1] → is a weight for each class, where the rare class is multiplied by a higher weight (α), and the common class is multiplied by a lower weight (1−α), this increases the loss contribution of the rare class and decreases the contribution of the dominant class, so the model pays more attention to the rare class during training.

* Gamma → responable for down weighting common/ easy examples to reduces the influence of easy examples on the loss function, resulting in more attention being paid to hard examples
To simulate sparse labeling, a binary supervision mask is applied, 1 → pixel contributes to loss and 0 → pixel ignored


In [ ]:
class SpatialFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, eps=1e-8):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        #self.reduction = reduction
        self.eps = eps

    def forward(self, logits, targets, mask):
        """
        logits: the logits from the netweok for each example
        mask: same shape as target, 0 --> ignore this pixel, 1--> don't 
        targets: the tensor of the binary classification label for each pixel in the inputs (0 for the background class and 1 for the water class)
        (Note using logits for inputs instead of predictions/probabilities, because the torch function apply sigmoid)
        
        **** shapes **** 
        logits: (B, 1, H, W)
        targets: (B, C=1, H, W)
        mask:  (B, H, W)
        """

        # 1. focal loss per pixel
        focal = sigmoid_focal_loss(logits.squeeze(1),
                                   targets,
                                   alpha= self.alpha,
                                   gamma= self.gamma,
                                   reduction= "none"
                                  )

        # 2. apply mask
        focal = focal * mask.float()

        # 3. average over labeled pixels only
        return focal.sum() / (mask.sum() + self.eps)
        
    def step(self, epoch):
        # optional
        if epoch > 10:
            self.gamma = 2.0 
print("Done")

### Model Evaluation using 2 approaches
The case is in the flood segmentation datasets often contain images with highly varying flood areas — some images contain large flood regions, while others contain only a few flood pixels. This imbalance can bias evaluation if a single metric is used, so, to ensure a fair assessment of model performance will use 2 approaches

1. Global pixel wise evaluation:
  *  This approach treats each pixel equally, making it suitable for evaluating the model as a pixel level binary classifier, to measure the overall segmentation performance across the entire evaluation dataset, will compute the following metrics globally:
*  Recall, Percision, F1 Score using the total of TP, FP, and FN aggregated over all pixels in all images of the evaluation set


2. Per image spatial evaluation
* To prevent images with large flood regions from dominating the evaluation, will compute Dice Score, and Intersection over Union (IoU) for each image individually, and then take the mean across all evaluation images
* This approach ensures each image contributes equally to the final metric and performance is fairly evaluated across images with small and large flood areas and make sure to assess the spatial overlap quality, since Dice and IoU are spatial overlap metrics, they directly reflect the quality of segmentation boundaries and region prediction per image.



In [ ]:
"""def get_predictions(model, dataloader, threshold=0.5):

    #return >>
      #  predictions_list: list of tensors (N, H, W)
      #  targets_list : list of tensors (N, H, W)
    
    model.eval()
    predictions_list = []
    targets_list = []

    with torch.no_grad():
        for batch in dataloader:
            imgs = batch["img"].to(device)
            targets = batch["target"].to(device)  # (B, H, W)

            logits = model(imgs)
            probs = torch.sigmoid(logits)
            predictions = (probs > threshold).float().squeeze(1)

            predictions_list.append(predictions)
            targets_list.append(targets)

    return torch.cat(predictions_list), torch.cat(targets_list)
"""

def evaluate_model_per_image(predictions, targets, eps=1e-7):
    
    predictions_sum = predictions.sum(dim=(1, 2))
    targets_sum = targets.sum(dim=(1, 2))
    
    intersection = (predictions * targets).sum(dim=(1, 2))
    union = predictions_sum + targets_sum - intersection

    iou = ((intersection + eps) / (union + eps))#.mean().item()
    dice = ((2 * intersection + eps) / (predictions_sum + targets_sum + eps))#.mean().item()

    return {"iou_per_image": iou,
            "dice_per_image": dice
           }


"""def evaluate_model_global(predictions, targets, eps=1e-7):

    # flatten everything
    predictions = predictions.view(-1)
    targets = targets.view(-1)

    TP = (predictions * targets).sum()
    FP = (predictions * (1 - targets)).sum()
    FN = ((1 - predictions) * targets).sum()
    TN = ((1 - predictions) * (1 - targets)).sum()

    precision = (TP / (TP + FP + eps)).item()
    recall = (TP / (TP + FN + eps)).item()
    f1 = (2 * precision * recall) / (precision + recall + eps)
    accuracy = ((TP + TN) / (TP + TN + FP + FN + eps)).item()

    iou = (TP / (TP + FP + FN + eps)).item()
    dice = ((2 * TP) / (2 * TP + FP + FN + eps)).item()

    return {
        "precision_global": precision,
        "recall_global": recall,
        "f1_global": f1,
        "accuracy_global": accuracy,
        "iou_global": iou,
        "dice_global": dice
    }"""


# Model Architecture
The segmentation model is: U-Net with Xception encoder (ImageNet pretrained)
* Encoder weights initialized from ImageNet
* Output channel: 1 (binary segmentation)
* Sigmoid activation applied during evaluation

# Optimization
* Optimizer: Adam
* Gradients monitored for NaN / Inf for numerical stability

# Training Strategy
* Due to the relatively small dataset size (290 images), a K-Fold Cross Validation strategy was adopted to ensure robust and unbiased model evaluation and more model generalization
* Number of folds: K = 5, each fold: 80% used for training and 20% used for validation

# Training Loop
For each fold:
* Initialize model
* Train for num_epoch
* Compute final validation metrics on held-out fold
* Store metrics
* Repeat for next fold



In [ ]:
def train_one_epoch(epoch_num, train_dataloader, model, optimizer, loss_function, device):
    #train_start_time = time.time()
    model.train()
    
    total_train_loss = 0

    inf_grad_count = 0
    nan_grad_count = 0
    
    #for batch_idx, batch in enumerate(train_dataloader): 
    for batch_idx, batch in enumerate(tqdm(train_dataloader)):

        # _________ Load Train Data _________
        imgs = batch["img"].to(device)
        targets = batch["target"].to(device)
        masks = batch["mask"].to(device)
        
        # _________ Training _________
        outputs_logits = model(imgs)
        loss = loss_function(outputs_logits, targets, masks)
        total_train_loss += loss.item()
        #print(f"Epoch: {epoch_num}, Batch: {batch_idx+1}, Train Loss: {loss.item()}")
        
        optimizer.zero_grad()
        loss.backward()

        # check nan/inf gradients
        for parm in model.parameters():
            if parm.grad is not None and (torch.isinf(parm.grad).any()):
                inf_grad_count += 1
                print(f"Found in Epoch: {epoch_num+1}, Batch: {batch_idx+1}, Total inf gradients: {inf_grad_count}")
            elif parm.grad is not None and (torch.isnan(parm.grad).any()):
                nan_grad_count += 1
                print(f"Found in Epoch: {epoch_num+1}, Batch: {batch_idx+1}, Total nan gradients: {nan_grad_count}")
                
        optimizer.step()
        
        
    #train_time = time.time() - train_start_time
    train_avg_loss = total_train_loss / len(train_dataloader)
    #print("Train one epoch time: ", train_time)
    return model, train_avg_loss

print("done ^^")

In [ ]:
def validate(model, threshold, device, loss_function, valid_dataloader):
    # _________ Validion _________
    total_vaild_loss = 0
    
    all_iou = []
    all_dice = []
    
    precision_metric = BinaryPrecision(threshold=threshold).to(device)
    recall_metric = BinaryRecall(threshold=threshold).to(device)
    f1_metric = BinaryF1Score(threshold=threshold).to(device)

    valid_start_time = time.time()
    model.eval()    
    with torch.no_grad():
        for batch_idx, batch in enumerate(valid_dataloader): 
            # _________ Load Val Data _________
            imgs = batch["img"].to(device)
            targets = batch["target"].to(device)
            masks = batch["mask"].to(device)
            
            outputs_logits = model(imgs)
            probs = torch.sigmoid(outputs_logits).squeeze(1)

            precision_metric.update(probs, targets)
            recall_metric.update(probs, targets)
            f1_metric.update(probs, targets)
            
            predictions = (probs > threshold).float()
            metrics = evaluate_model_per_image(predictions, targets)
            all_iou.append(metrics["iou_per_image"]) #metrics["iou_per_image"] --> (B,)
            all_dice.append(metrics["dice_per_image"])  #(B,)

            loss = loss_function(outputs_logits, targets, masks)
            total_vaild_loss += loss.item()
            
            if batch_idx == 0:
                img_visualization({"Image":imgs[0], "Mask":masks[0], "Target": targets[0], "Output": predictions[0]}, "Validation phase")
        
        avg_val_loss = total_vaild_loss/len(valid_dataloader)
        iou_score = torch.cat(all_iou).mean().item()
        dice_score = torch.cat(all_dice).mean().item()
        
        precision = precision_metric.compute().item()
        recall = recall_metric.compute().item()
        f1 = f1_metric.compute().item()
        
        precision_metric.reset()
        recall_metric.reset()
        f1_metric.reset()
        
        valid_time = time.time() - valid_start_time
        print("Validation time: ",valid_time)
        return avg_val_loss, iou_score, dice_score, precision, recall, f1

# Final Reporting
For each drop percentage experiment, the following are reported:
* → Precision (global)
* → Recall (global)
* → F1 Score (global)
* → IoU (per image mean)
* → Dice (per image mean)
Final performance across all folds is reported as: Metric: Mean ± Std

# Experiments configurations:
* Image size: 224,224
* Number of epochs: 10
* Batch size: 16
* Threshold: 0.5
* Mask strategy: Random 
* No data augmentation is applied 
* Alpha value for focal loss:0.25
* Gamma value for focal loss: for the first experiment 2, and for the second 5



In [ ]:
def run_experiment(drop, train_dataframe, config):

    # ---------- unpack config ----------
    img_size = config["img_size"]
    batch_size = config["batch_size"]
    num_workers = config["num_workers"]
    pin_memory = config["pin_memory"]
    learning_rate = config["learning_rate"]
    n_splits = config["n_splits"]
    num_epoch = config["num_epoch"]
    threshold = config["threshold"]
    #patience_epochs = config["patient_epoches"]
    root_path = config["root_path"]
    mask_strategy = config["mask_strategy"]
    augmentation_mode = config["augmentation_mode"]
    loss_gamma = config["loss_gamma"]
    loss_alpha = config["loss_alpha"]
    encoder = config["encoder"]
    encoder_trained_ds = config["encoder_trained_ds"]
    patience = 3 #config["patience"]
    persistent_workers= True #config["persistent_workers"]
    prefetch_factor =  2  if num_workers > 0 else None #config["prefetch_factor"]

    # ---------- metrics lists  ----------
    iou_scores = []
    dice_scores = []
    recall_scores = []
    precision_scores = []
    f1_score_scores = []
    # ---------- KFold ----------
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # ---------- folds loop ----------
    for fold, (train_index, val_index) in enumerate(kfold.split(train_dataframe)):
        print(f"===== Fold {fold + 1}/{n_splits} === Encoder: {encoder} with drop: {drop} and gamma: {loss_gamma}=====")

        train_df = train_dataframe.iloc[train_index]
        val_df = train_dataframe.iloc[val_index]

        # ---------- datasets/dataloaders ----------
        train_dataset = READ_FLOOD_AREA_DATA_SET(df=train_df,root_url=root_path,img_size=img_size,
                                                 drop_pixels_percentage=drop,augmentation_mode=augmentation_mode,
                                                 mode="train",mask_strategy=mask_strategy)

        val_dataset = READ_FLOOD_AREA_DATA_SET(df=val_df,root_url=root_path,img_size=img_size,
                                               drop_pixels_percentage=drop,
                                               mode="train",mask_strategy=mask_strategy)

        train_loader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True,num_workers=num_workers,
                                  pin_memory=pin_memory,
                                 persistent_workers=persistent_workers,
                                 prefetch_factor=prefetch_factor)

        val_loader = DataLoader(val_dataset,batch_size=batch_size,shuffle=False,num_workers=num_workers,
                                pin_memory=pin_memory,
                               persistent_workers=persistent_workers,
                               prefetch_factor=prefetch_factor)

        # ---------- model ----------
        model = smp.Unet(encoder_name=encoder,encoder_weights=encoder_trained_ds,in_channels=3,classes=1).to(device)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        loss_fn = SpatialFocalLoss(gamma = loss_gamma, alpha = loss_alpha)

        best_model_state = model.state_dict()
        best_val_loss = float("inf")
        patience_counter = 0
        best_metrics = None
        
        # ---------- training loop ----------
        for epoch in range(1, num_epoch + 1):
            print(f"Epoch {epoch}/{num_epoch}")
            model, train_loss = train_one_epoch(epoch, train_loader,model, optimizer,loss_fn, device)

        # ---------- evaluation ----------
            val_loss, iou, dice, precision,recall,f1 = validate(model, threshold, device, loss_fn, val_loader)
        
        # ---------- early stopping ----------
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict()
                best_metrics =  (val_loss, iou, dice, precision, recall, f1) 
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print("Early stopping triggered")
                break;
            
        model.load_state_dict(best_model_state)
        val_loss, iou, dice, precision,recall,f1 = best_metrics
        
        iou_scores.append(iou)
        dice_scores.append(dice)
        recall_scores.append(recall)
        f1_score_scores.append(f1)
        precision_scores.append(precision)
        
        print(f"Fold {fold + 1} | "f"Train Loss: {train_loss:.4f} | "f"Val Loss: {val_loss:.4f} ")
        
    print(f"===== Final Cross Validation Results For Drop: {drop} =====")
    print(f"Precision:   {np.mean(precision_scores):.4f} ± {np.std(precision_scores):.4f}")
    print(f"Recall:   {np.mean(recall_scores):.4f} ± {np.std(recall_scores):.4f}")
    print(f"F1 Score:   {np.mean(f1_score_scores):.4f} ± {np.std(f1_score_scores):.4f}")
    print(f"IoU:        {np.mean(iou_scores):.4f} ± {np.std(iou_scores):.4f}")
    print(f"Dice:       {np.mean(dice_scores):.4f} ± {np.std(dice_scores):.4f}")
    
    return {drop:[iou_scores, dice_scores, recall_scores, f1_score_scores, precision_scores]}

In [ ]:
# Common Configurations:
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)
random_state = 42
drop_pixels_percentage_list = [0.1, 0.2, 0.3, 0.5, 0.7]
df_path = "/kaggle/input/flood-area-segmentation/metadata.csv"
img_mask_mapping_df = pd.read_csv(df_path)

### Encoder: Resnet, Gamma: 2, Drop Lists: [0.1, 0.2, 0.3, 0.5, 0.7]

In [ ]:
# ------------------ [1]------------------
conf_dict_resnet_1 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             #"patient_epoches": 5,
             "mask_strategy": "random",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 2.0,
             "encoder": "resnet18",
             "encoder_trained_ds":"imagenet"
}


for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_resnet_1)

In [ ]:
# ------------------ [1]------------------
conf_dict_efficient_1 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             #"patient_epoches": 5,
             "mask_strategy": "random",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 2.0,
             "encoder": "efficientnet-b0",
             "encoder_trained_ds":"imagenet"
}


for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_efficient_1)

# Expirment 1, using Ecoder:xception, Random Sampling & Drop using [0.1, 0.3, 0.5, 0.7]

In [ ]:
# ------------------ [2]------------------

conf_dict_xception_1 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             #"patient_epoches": 5,
             "mask_strategy": "random",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 2.0,
             "encoder": "xception",
             "encoder_trained_ds":"imagenet"
}

for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_xception_1)

# Experiment 1 Conclusion:
Increasing drop percentage generally
* → Slightly increases precision
* → Decreases recall
* → Reduces spatial metrics (IoU, Dice)
This suggests:
* → High drop acts as regularization
* → Excessive pixel dropping reduces spatial learning consistency.

##### Although drop=0.7 achieves slightly better quantitative metrics (IoU, Dice, F1), visual inspection suggests smoother and less sharp boundaries compared to drop=0.5. This indicates that stronger pixel dropping acts as regularization, improving region coverage while slightly reducing boundary precision.

##### For higher drop ratios (0.5 and 0.7), the model successfully captures large flood regions. However, it tends to misclassify small flood areas located between objects such as buildings or narrow streets between dense trees.

# Expemint 2 with Gamma = 5

In [ ]:
# ------------------ [3] ------------------

conf_dict_xception_2 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             "mask_strategy": "random",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 5.0 ,
             "encoder": "xception",
             "encoder_trained_ds":"imagenet"}



df_path = "/kaggle/input/flood-area-segmentation/metadata.csv"
img_mask_mapping_df = pd.read_csv(df_path)

for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_xception_2)

# Observations from experiment 1, and experiment 2
* → Increasing gamma from 2 → 5 resulted in:
* Lower Recall across all drop values
* Lower IoU and Dice
* Lower F1 Score
Increasing gamma to 5 resulted in a consistent decrease across all evaluation metrics compared to gamma=2. Although focal loss with higher gamma emphasizes hard-to-classify pixels, the model became overly focused on difficult samples, reducing its overall segmentation performance. Given the relatively small dataset size, gamma=2 provided a better balance between hard example mining and stable learning.

* experiment 1 loss example from drop=0.7 and fold 5:
* * Train Loss: 0.0209 | Val Loss: 0.0357 
* experiment 2 loss example from drop=0.7 and fold 5:
* * Train Loss: 0.0035 | Val Loss: 0.0052 

Even though focal loss became numerically smaller, segmentation quality decreased.
High gamma overly emphasizes hard pixels and may destabilize learning and reduce generalization in moderately imbalanced datasets


# Experiment 3: Using Balanced Samplping mask  

In [ ]:
# ------------------ [4]------------------

conf_dict_xception_3 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             "mask_strategy": "balanced",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 2.0,
             "encoder": "xception",
             "encoder_trained_ds":"imagenet"}


for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_xception_3)

In [ ]:
# ------------------ [5]------------------
conf_dict_xception_4 = {"img_size": (224, 224),
             "num_workers": 2,
             "pin_memory": True,
             "learning_rate": 1e-4,
             "n_splits": 3,
             "num_epoch": 5,
             "batch_size": 16,
             "threshold": 0.5,
             "mask_strategy": "balanced",
             "root_path": "/kaggle/input/flood-area-segmentation/",
             "augmentation_mode": False,
             "loss_alpha" : 0.25, 
             "loss_gamma" : 2.0,
             "encoder": "xception",
             "encoder_trained_ds":"imagenet"}

for drop in drop_pixels_percentage_list:
    run_experiment(drop,img_mask_mapping_df,conf_dict_xception_4)
    